# 🎯 Análisis Final: Aprendizaje No Supervisado — FIDE Chess Dataset

**Evaluación 2 — Aprendizaje No Supervisado (30%)**

Este notebook **no entrena modelos desde cero**. Carga los resultados ya
calculados por el pipeline de Kedro (`clustering_metrics` y `unsupervised_model`)
y genera visualizaciones:
1. Gráfico del Método del Codo (inercias precalculadas)
2. Scatter plot 2D con PCA (coloreado por cluster)
3. Métricas de clustering y distribución

In [ ]:
# ============================================================
# Celda 1: Inicializar sesión de Kedro
# ============================================================
%load_ext kedro.ipython

In [ ]:
# ============================================================
# Celda 2: Imports y configuración de visualización
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Estilo profesional
sns.set_theme(style='whitegrid', palette='viridis', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
print('✅ Librerías cargadas correctamente')

In [ ]:
# ============================================================
# Celda 3: Cargar reportes y modelo desde el catálogo
# ============================================================
# Reporte con inercias, silhouettes, PCA variance, etc.
cluster_report = catalog.load('clustering_metrics')

# Modelo KMeans entrenado (para obtener labels de los datos)
kmeans_model = catalog.load('unsupervised_model')

# Datos preprocesados (para recalcular PCA 2D — las coordenadas
# no se almacenan en el JSON por su gran tamaño)
df = catalog.load('fide_preprocessed_data')

print('📋 Reportes y modelo cargados exitosamente')
print(f'   Mejor K:              {cluster_report["best_k"]}')
print(f'   Silhouette Score:     {cluster_report["silhouette_score"]:.4f}')
print(f'   Calinski-Harabasz:    {cluster_report["calinski_harabasz_score"]:.4f}')
print(f'   Davies-Bouldin:       {cluster_report["davies_bouldin_score"]:.4f}')
print(f'   Distribución:         {cluster_report["cluster_distribution"]}')

---
## 1. Método del Codo — Inercias Precalculadas

In [ ]:
# ============================================================
# GRÁFICO 1: Método del Codo + Silhouette Score
# ============================================================
# Extraer datos del codo directamente del reporte
elbow = cluster_report['elbow_data']
k_values = elbow['k_values']
inertias = elbow['inertias']
silhouettes = elbow['silhouettes']
best_k = cluster_report['best_k']

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- Panel izquierdo: Método del Codo (Inercia) ---
axes[0].plot(k_values, inertias, 'o-', color='#3498db',
             linewidth=2.5, markersize=9, markerfacecolor='white',
             markeredgewidth=2, markeredgecolor='#3498db')
axes[0].axvline(best_k, color='#e74c3c', linestyle='--', linewidth=2,
                label=f'K óptimo = {best_k}', alpha=0.8)

# Sombrear el punto óptimo
best_idx = k_values.index(best_k)
axes[0].scatter([best_k], [inertias[best_idx]], color='#e74c3c',
                s=200, zorder=5, edgecolors='white', linewidths=2)

axes[0].set_title('Método del Codo', fontweight='bold', fontsize=15)
axes[0].set_xlabel('Número de Clusters (K)')
axes[0].set_ylabel('Inercia (WCSS)')
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(k_values)

# --- Panel derecho: Silhouette Score ---
axes[1].plot(k_values, silhouettes, 's-', color='#2ecc71',
             linewidth=2.5, markersize=9, markerfacecolor='white',
             markeredgewidth=2, markeredgecolor='#2ecc71')
axes[1].axvline(best_k, color='#e74c3c', linestyle='--', linewidth=2,
                label=f'K óptimo = {best_k}', alpha=0.8)

# Sombrear el punto óptimo
axes[1].scatter([best_k], [silhouettes[best_idx]], color='#e74c3c',
                s=200, zorder=5, edgecolors='white', linewidths=2)

axes[1].set_title('Silhouette Score por K', fontweight='bold', fontsize=15)
axes[1].set_xlabel('Número de Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(k_values)

plt.suptitle('Selección del Número Óptimo de Clusters',
             fontsize=16, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

# Tabla complementaria
print('\n📊 Datos del análisis de codo:')
elbow_df = pd.DataFrame({
    'K': k_values,
    'Inercia': [f'{i:,.2f}' for i in inertias],
    'Silhouette': silhouettes,
})
display(elbow_df.set_index('K'))

---
## 2. Scatter Plot 2D — Proyección PCA con Clusters

In [ ]:
# ============================================================
# GRÁFICO 2: Scatter plot 2D con PCA coloreado por cluster
# ============================================================
# Reproducir la misma transformación del pipeline para obtener
# las coordenadas 2D (PCA). El modelo KMeans ya está entrenado.
CLUSTER_FEATURES = ['rating_std_avg', 'rating_change',
                     'total_months_active', 'age_approx']
available = [c for c in CLUSTER_FEATURES if c in df.columns]
X = df[available].dropna()

# Escalar (misma transformación que el pipeline)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Obtener etiquetas de cluster del modelo cargado
cluster_labels = kmeans_model.predict(X_scaled)

# PCA a 2 componentes
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Varianza explicada (validar con el reporte)
pca_var = cluster_report['pca_variance_explained']
print(f'Varianza explicada (reporte): PC1={pca_var["PC1"]:.4f}, PC2={pca_var["PC2"]:.4f}, Total={pca_var["total"]:.4f}')
print(f'Varianza explicada (PCA local): PC1={pca.explained_variance_ratio_[0]:.4f}, PC2={pca.explained_variance_ratio_[1]:.4f}')

# --- Gráfico ---
fig, ax = plt.subplots(figsize=(14, 10))

scatter = ax.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=cluster_labels,
    cmap='viridis',
    alpha=0.4,
    s=8,
    edgecolors='none',
)

# Centroides proyectados en PCA
centroids_pca = pca.transform(kmeans_model.cluster_centers_)
ax.scatter(
    centroids_pca[:, 0], centroids_pca[:, 1],
    c='red', marker='X', s=300, edgecolors='black', linewidths=2,
    label='Centroides', zorder=10,
)

# Etiquetas de centroides
for i, (cx, cy) in enumerate(centroids_pca):
    ax.annotate(f'C{i}', (cx, cy), fontsize=12, fontweight='bold',
                ha='center', va='bottom', color='darkred',
                xytext=(0, 12), textcoords='offset points')

ax.set_title(f'Clustering de Jugadores FIDE (K={best_k}) — Proyección PCA 2D',
             fontweight='bold', fontsize=15)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} varianza)', fontsize=13)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} varianza)', fontsize=13)
ax.legend(fontsize=12, loc='upper right')
ax.grid(True, alpha=0.2)

cbar = plt.colorbar(scatter, ax=ax, label='Cluster', shrink=0.8)
cbar.ax.tick_params(labelsize=10)

plt.tight_layout()
plt.show()

---
## 3. Métricas de Evaluación del Clustering

In [ ]:
# ============================================================
# Tabla de métricas con interpretación
# ============================================================
metrics_df = pd.DataFrame({
    'Métrica': ['Silhouette Score', 'Calinski-Harabasz Index', 'Davies-Bouldin Index'],
    'Valor': [
        cluster_report['silhouette_score'],
        cluster_report['calinski_harabasz_score'],
        cluster_report['davies_bouldin_score'],
    ],
    'Interpretación': [
        'Más alto = mejor (rango -1 a 1)',
        'Más alto = mejor separación entre clusters',
        'Más bajo = mejor (clusters más compactos)',
    ]
})

print(f'📊 Métricas de Clustering (K={best_k}):')
display(metrics_df.style.format({'Valor': '{:.4f}'}).hide(axis='index'))

In [ ]:
# ============================================================
# Distribución de jugadores por cluster
# ============================================================
cluster_dist = cluster_report['cluster_distribution']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pie chart
labels = [k.replace('cluster_', 'Cluster ') for k in cluster_dist.keys()]
sizes = list(cluster_dist.values())
colors = sns.color_palette('viridis', n_colors=len(labels))

axes[0].pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors,
            startangle=90, pctdistance=0.85,
            wedgeprops=dict(width=0.5, edgecolor='white', linewidth=2))
axes[0].set_title('Distribución de Jugadores por Cluster', fontweight='bold')

# Bar chart
bars = axes[1].bar(labels, sizes, color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, sizes):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[1].set_title('Cantidad de Jugadores por Cluster', fontweight='bold')
axes[1].set_ylabel('Cantidad')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Conclusiones Generales del Proyecto

### Evaluación 1 — Calidad y Transformación de Datos
- Se cargaron exitosamente 4 datasets FIDE (~12M registros).
- Se limpiaron duplicados, nulos y outliers (IQR en rating_standard).
- Se integraron las tablas mediante joins y se crearon features derivadas.
- La validación post-transformación confirma la integridad del dataset final.

### Evaluación 2 — Machine Learning
- **Supervisado:** Se entrenaron 5 modelos de clasificación para predecir si un jugador es "experto" (ELO > 2000).
- **Evaluación:** Se aplicó validación cruzada 5-fold con múltiples métricas.
- **Optimización:** GridSearchCV / RandomizedSearchCV mejoraron los hiperparámetros.
- **No Supervisado:** K-Means reveló segmentos naturales de jugadores basados en rating, actividad y edad.

### Lecciones Aprendidas
- La modularidad de Kedro facilita la reproducibilidad y el mantenimiento del código.
- La combinación de técnicas supervisadas y no supervisadas enriquece el análisis.
- La semilla fija (`random_state=42`) garantiza resultados reproducibles.